# JK Assistance - admission samples

Sample students who have just finished 12th grade, run through the admission engine:

1. Entry tests: maximum and minimum marks (MDCAT out of 180, ECAT out of 400)
2. Eligibility checklists: required vs obtained, for sample students
3. MBBS merit aggregate (PM&DC formula 50 / 40 / 10)
4. Edge cases: below the minimum, test not yet taken, a failed ECAT with SEET still open, the Mathematics requirement, age limit
5. Step-by-step "how to apply" for UAJK, MUST and MBBS (the JAC website is down, so working channels are used)
6. Admission dates
7. The chat assistant on sample questions (English and Roman Urdu)
8. The MCP server, called through the official client

"Today" is fixed at **2026-09-29** so results are reproducible.

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ["JK_TODAY"] = "2026-09-29"
USE_CLAUDE = bool(os.environ.get("ANTHROPIC_API_KEY"))
if not USE_CLAUDE:
    os.environ["JK_DISABLE_LLM"] = "1"
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Markdown, display
pd.set_option("display.max_colwidth", 80)
from jk_assistance import advisor, assistant
from jk_assistance.knowledge import get_knowledge
kb = get_knowledge()
print(f"{len(kb.programs)} programs, {len(kb.tests)} entry tests, {len(kb.dated_deadlines)} dated admission events, {len(kb.chunks)} evidence chunks")

40 programs, 5 entry tests, 14 dated admission events, 1443 evidence chunks


## 1. Entry tests - maximum and minimum marks

In [2]:
rows = []
for tid, t in kb.tests.items():
    for k, v in (t.get("passing") or {}).items():
        if isinstance(v, dict):
            rows.append({"test": t["name"], "max marks": t.get("max_marks") or "on result card", "minimum for": k,
                         "min %": v["percent"], "min marks": v.get("marks") or (round(v["percent"] / 100 * t["max_marks"]) if t.get("max_marks") else "-")})
pd.DataFrame(rows)

,test,max marks,minimum for,min %,min marks
0,MDCAT (Medical & Dental Colleges Admission Test),180,mbbs,55,99
1,MDCAT (Medical & Dental Colleges Admission Test),180,bds,50,90
2,ECAT (UET Engineering College Admission Test),400,pec_engineering,33,132
3,MUST SEET (State Entry Examination Test),on result card,engineering,33,-
4,UAJK Entry Test,on result card,general,50,-
5,HEC Law Admission Test (LAT),on result card,uajk_llb,50,-


## 2. Eligibility checklists for sample students

In [3]:
def show(profile, n=6):
    r = advisor.advise(profile)
    for p in r["programs"][:n]:
        print(f"\n{p['program'][:70]}  [{p['institution'].upper()}]  -> {p['eligibility']}  | chance: {p['chance']}")
        display(pd.DataFrame(p["checks"])[["criterion", "required", "yours", "status"]])
    return r

print("Sample A - FSc pre-medical 946/1100, matric 1010/1100, MDCAT 140/180, wants medicine")
r = show({"group": "pre_med", "hssc": "946/1100", "ssc": "1010/1100", "tests": {"mdcat": 140}, "state_subject": True, "interests": ["medicine"]}, n=3)

Sample A - FSc pre-medical 946/1100, matric 1010/1100, MDCAT 140/180, wants medicine

MBBS - AJK Medical College Muzaffarabad / MBBS Medical College Mirpur   [JAC]  -> eligible  | chance: merit aggregate 82.472% - public-college merit is very competitive and closing merits are not published; apply to all options


,criterion,required,yours,status
0,Intermediate group,"F.Sc. Pre-Medical, A-level (with IBCC equivalence)",F.Sc. Pre-Medical,pass
1,HSSC marks,≥ 60%,86%,pass
2,MDCAT (Medical & Dental Colleges Admission Test),≥ 55% = 99 / 180,140 / 180 = 77.78%,pass
3,AJK State Subject (Class-I),required for AJK-national seats,yes,pass



MBBS - Federal Medical College Islamabad (SZABMU)  [SZABMU]  -> check_required  | chance: merit aggregate 82.472% - public-college merit is very competitive and closing merits are not published; apply to all options


,criterion,required,yours,status
0,Intermediate group,"F.Sc. Pre-Medical, A-level (with IBCC equivalence)",F.Sc. Pre-Medical,pass
1,HSSC marks,≥ 60%,86%,pass
2,MDCAT (Medical & Dental Colleges Admission Test),≥ 55% = 99 / 180,140 / 180 = 77.78%,pass
3,Domicile + proof of residence,AJK domicile for the AJK-quota seats,bring proof,check



MBBS - Shaikh Khalifa Bin Zayed Al-Nahyan Medical & Dental College Lah  [SZABMU]  -> check_required  | chance: merit aggregate 82.472% - public-college merit is very competitive and closing merits are not published; apply to all options


,criterion,required,yours,status
0,Intermediate group,"F.Sc. Pre-Medical, A-level (with IBCC equivalence)",F.Sc. Pre-Medical,pass
1,HSSC marks,≥ 60%,86%,pass
2,MDCAT (Medical & Dental Colleges Admission Test),≥ 55% = 99 / 180,140 / 180 = 77.78%,pass
3,Domicile + proof of residence,AJK domicile for the AJK-quota seats,bring proof,check


In [4]:
print("Sample B - FSc pre-engineering 880/1100, ECAT 210/400, wants engineering")
r = show({"group": "pre_eng", "hssc": "880/1100", "tests": {"ecat": 210}, "interests": ["engineering"]}, n=3)

Sample B - FSc pre-engineering 880/1100, ECAT 210/400, wants engineering

B.Sc. Engineering - Electrical / Mechanical / Computer Systems / Softw  [MUST]  -> eligible  | chance: ECAT 210/400 is above the Fall 2026 average for your group (203/400) - heuristic


,criterion,required,yours,status
0,Intermediate group,"F.Sc. Pre-Engineering, F.Sc. Pre-Medical, ICS (Intermediate in Computer Scie...",F.Sc. Pre-Engineering,pass
1,HSSC marks,≥ 60%,80%,pass
2,MUST SEET (State Entry Examination Test),≥ 33%,not entered / not taken yet,pending
3,ECAT (UET Engineering College Admission Test),≥ 33% = 132 / 400,210 / 400 = 52.5%,pass



B.Sc. Civil Engineering  [MUST]  -> eligible  | chance: ECAT 210/400 is above the Fall 2026 average for your group (203/400) - heuristic


,criterion,required,yours,status
0,Intermediate group,"F.Sc. Pre-Engineering, F.Sc. Pre-Medical, ICS (Intermediate in Computer Scie...",F.Sc. Pre-Engineering,pass
1,HSSC marks,≥ 60%,80%,pass
2,MUST SEET (State Entry Examination Test),≥ 33%,not entered / not taken yet,pending
3,ECAT (UET Engineering College Admission Test),≥ 33% = 132 / 400,210 / 400 = 52.5%,pass



B.Sc. Electrical Engineering (PEC)  [UAJK]  -> eligible_after_test  | chance: good (heuristic)


,criterion,required,yours,status
0,Intermediate group,"F.Sc. Pre-Engineering, ICS (Intermediate in Computer Science), DAE (Diploma ...",F.Sc. Pre-Engineering,pass
1,HSSC marks,≥ 60%,80%,pass
2,UAJK Entry Test,≥ 50%,not entered / not taken yet,pending


In [5]:
print("Sample C - ICS 704/1100 (64%), no test yet, no interest given -> which fields are open?")
r = advisor.advise({"group": "ics", "hssc": "704/1100"})
display(pd.DataFrame(r["fields_open_to_you"]))
pd.DataFrame([{"program": p["program"][:60], "inst": p["institution"].upper(), "status": p["eligibility"], "chance": p["chance"],
               "1st sem fee": (p["fees"] or {}).get("first_semester")} for p in r["programs"] if p["eligibility"] != "not_eligible"])

Sample C - ICS 704/1100 (64%), no test yet, no interest given -> which fields are open?


,field,label,programs
0,computing,Computing / IT / AI,5
1,social_sciences,Social sciences & languages,5
2,sciences,Natural sciences,4
3,engineering,Engineering,3
4,engineering_technology,Engineering Technology,2
5,business,Business & commerce,2
6,law,Law,2
7,education,Education,2
8,arts,Arts & design,1


,program,inst,status,chance,1st sem fee
0,BS Physics / Chemistry / Mathematics / Statistics,MUST,eligible,strong (heuristic),50335.0
1,BS Environmental Science / BS Forestry,MUST,eligible,strong (heuristic),NaN
2,BS Economics / Islamic Studies / International Relations / S,MUST,eligible,strong (heuristic),45460.0
3,BS English,MUST,eligible,strong (heuristic),49590.0
4,BS Home Economics / Fashion and Design,MUST,eligible,strong (heuristic),44910.0
5,BS Education,MUST,eligible,strong (heuristic),41260.0
6,BBA / BS Accounting & Finance / BS Banking & Finance / BS Co,MUST,eligible,strong (heuristic),NaN
7,BS Computer Science / BS Information Technology,MUST,eligible,moderate (heuristic),59970.0
8,BS Physics / Chemistry / Mathematics / Statistics / Statisti,UAJK,eligible_after_test,strong (heuristic),73430.0
9,BS English / Economics / Sociology / Kashmir Studies / Urdu,UAJK,eligible_after_test,strong (heuristic),65290.0


## 3. MBBS merit aggregate - PM&DC formula (MDCAT 50% + HSSC 40% + SSC 10%)

In [6]:
rows = []
for mdcat in [99, 120, 140, 160, 175]:
    r = advisor.advise({"group": "pre_med", "hssc": "946/1100", "ssc": "1010/1100", "tests": {"mdcat": mdcat}, "state_subject": True, "interests": ["medicine"]})
    m = next(p for p in r["programs"] if p["id"] == "jac.mbbs")
    rows.append({"MDCAT": f"{mdcat}/180", "MDCAT %": round(mdcat / 180 * 100, 2), "eligible": m["eligibility"], "merit aggregate %": m["merit_aggregate"]})
pd.DataFrame(rows)

,MDCAT,MDCAT %,eligible,merit aggregate %
0,99/180,55.00,eligible,71.082
1,120/180,66.67,eligible,76.917
2,140/180,77.78,eligible,82.472
3,160/180,88.89,eligible,88.027
4,175/180,97.22,eligible,92.192


## 4. Edge cases

In [7]:
cases = [
    ("MDCAT 95/180 -> below 99 for MBBS, but >= 90 for BDS", {"group": "pre_med", "hssc": 80, "tests": {"mdcat": 95}, "state_subject": True, "interests": ["medicine", "dentistry"]}, ["jac.mbbs", "jac.bds"]),
    ("HSSC 58% -> below the 60% needed for MBBS", {"group": "pre_med", "hssc": 58, "tests": {"mdcat": 150}, "state_subject": True, "interests": ["medicine"]}, ["jac.mbbs"]),
    ("Pre-medical, MDCAT not taken yet", {"group": "pre_med", "hssc": 80, "state_subject": True, "interests": ["medicine"]}, ["jac.mbbs"]),
    ("ECAT 120/400 (< 132) but MUST SEET not taken yet", {"group": "pre_eng", "hssc": 80, "tests": {"ecat": 120}, "interests": ["engineering"]}, ["must.engineering"]),
    ("Pre-medical wants UAJK BS CS without Mathematics", {"group": "pre_med", "hssc": 80, "interests": ["computing"], "institutions": ["uajk"]}, ["uajk.bs_cs_ai"]),
    ("Same, with additional Mathematics", {"group": "pre_med", "hssc": 80, "interests": ["computing"], "institutions": ["uajk"], "studied_math": True}, ["uajk.bs_cs_ai"]),
    ("Age 25 at UAJK (limit 23)", {"group": "ics", "hssc": 70, "age": 25, "interests": ["computing"], "institutions": ["uajk"]}, ["uajk.bs_cs_ai"]),
    ("FA 55% wants engineering", {"group": "fa", "hssc": 55, "interests": ["engineering"]}, ["must.engineering"]),
]
rows = []
for name, prof, ids in cases:
    r = advisor.advise(prof)
    for pid in ids:
        p = next(x for x in r["programs"] if x["id"] == pid)
        rows.append({"case": name, "program": pid, "result": p["eligibility"],
                     "why": "; ".join(f"{c['criterion']}: {c['yours']} vs {c['required']}" for c in p["checks"] if c["status"] in ("fail", "pending"))[:110]})
pd.DataFrame(rows)

,case,program,result,why
0,"MDCAT 95/180 -> below 99 for MBBS, but >= 90 for BDS",jac.mbbs,not_eligible,MDCAT (Medical & Dental Colleges Admission Test): 95 / 180 = 52.78% vs ≥ 55%...
1,"MDCAT 95/180 -> below 99 for MBBS, but >= 90 for BDS",jac.bds,eligible,
2,HSSC 58% -> below the 60% needed for MBBS,jac.mbbs,not_eligible,HSSC marks: 58% vs ≥ 60%
3,"Pre-medical, MDCAT not taken yet",jac.mbbs,eligible_after_test,MDCAT (Medical & Dental Colleges Admission Test): not entered / not taken ye...
4,ECAT 120/400 (< 132) but MUST SEET not taken yet,must.engineering,eligible_after_test,MUST SEET (State Entry Examination Test): not entered / not taken yet vs ≥ 3...
5,Pre-medical wants UAJK BS CS without Mathematics,uajk.bs_cs_ai,not_eligible,Intermediate Mathematics: not studied vs required; UAJK Entry Test: not ente...
6,"Same, with additional Mathematics",uajk.bs_cs_ai,eligible_after_test,UAJK Entry Test: not entered / not taken yet vs ≥ 50%
7,Age 25 at UAJK (limit 23),uajk.bs_cs_ai,not_eligible,UAJK Entry Test: not entered / not taken yet vs ≥ 50%; Age at closing date: ...
8,FA 55% wants engineering,must.engineering,not_eligible,Intermediate group: F.A. (Intermediate Arts / Humanities) vs F.Sc. Pre-Engin...


## 5. How to apply - step by step

In [8]:
display(Markdown("\n".join(assistant.render_howto(advisor.how_to_apply("uajk")))))

#### How to apply - The University of Azad Jammu & Kashmir (UAJK)
Portal: https://admissions.uajk.edu.pk/
1. **Check eligibility** - Match your HSSC group and marks with the program (see the results table).
2. **Apply online** - Create an account and fill the form at https://admissions.uajk.edu.pk/ (or collect a hard-copy form from the DSA office, Chellah Campus).
3. **Pay the challan** - Rs. 3,000 for the first program + Rs. 1,000 per additional program; entry test Rs. 1,000 + Rs. 200 per extra program is added automatically. Pay at any HBL branch. The FIRST challan must be paid - applications with only later challans paid are rejected.
4. **Send printed copies** - Online applicants also send a printed copy of each application with the challan copy and attested documents to the Directorate of Students' Affairs (or their campus).
5. **Entry test** - Appear at the campus you applied to (where a test applies; 50% to pass). No call letter is issued - watch the website.
6. **Merit list** - Merit lists are displayed on DSA / department notice boards and the website; no admission letter is issued.
7. **Interview and verification** - Bring ORIGINAL documents for verification (engineering/DPT also have an interview).
8. **Deposit fee and enrol** - Pay the 1st-semester fee by the date on the merit list. Self-support seats pay the normal fee + the self-support fee by demand draft to Director Finance, UAJK.
**Documents:**
- [ ] Attested copies of all educational certificates (SSC, HSSC result cards/certificates)
- [ ] Character certificate from the college/institute last attended
- [ ] NOC / migration certificate if HSSC is from a board other than BISE Mirpur AJ&K
- [ ] Copy of Domicile and State Subject certificate
- [ ] Copy of CNIC or Form-B
- [ ] Four recent passport-size photographs
- [ ] Copy of guardian's CNIC
- [ ] NOC from employer if employed (study leave for full-time study)
- [ ] Nomination/certificate for any reserved-seat category
**Dates:** 2026-10-05 (6 days) UAJK Fall 2026 admissions - last date to apply (extended); 2026-10-05 (6 days) UAJK merit scholarship / tuition rebate 2026-27 - apply by
**Age limit:** 23 years for BS programs (Prospectus 2022-23 - confirm in the current advertisement)
**Reserved seats:** Per department: open merit 31, self-support 6, armed forces children 2 (GHQ), school-teacher child 1, college-teacher child 1, university-employee child 1, university-teacher child 1, disabled 1, sports 1, Gilgit-Baltistan 4, FATA 2, Punjab 1, KPK 1, Sindh 1, Balochistan 1, AJK overseas (OPF) 1. Nominations must reach DSA before the advertised date.
**Scholarships:** Merit rebate 2026-27: top 3 students in each undergraduate program get 50% / 40% / 30% tuition rebate. Need-based: HEC need-based, PEEF, Muslim Hands, HBL Foundation via the Directorate of Student Financial Assistance.
**Fee rules:** Fees shown are for sessions starting 2025 (UAJK notification of 12 Sept 2025); UAJK notes fees typically rise about 10% per year. From the 2nd semester, fee is due by the end of week 3 of classes; 10% fine from week 4 to week 6.
**Contact:** Admissions help (Directorate of Students' Affairs) 05822-960440; DSA Chellah Campus 05822-960463; DSA King Abdullah Campus 05822-962324; Scholarships (Student Financial Assistance) 05822-960410

In [9]:
display(Markdown("\n".join(assistant.render_howto(advisor.how_to_apply("jac")))))

#### How to apply - Joint Admission Committee (JAC) AJ&K - MBBS/BDS in AJK public medical & dental colleges
> ⚠️ NOT WORKING - returns 403 / Cloudflare block (checked 2026-09-29). Use the working channels below.
> - PM&DC (national regulator: eligibility, MDCAT passing marks, merit formula) - https://pmdc.pk/
> - SZABMU Islamabad (conducts MDCAT for AJK; federal colleges with AJK-quota seats) - https://www.szabmu.edu.pk/admission/mbbs-admission
> - Poonch Medical College admission cell - 05824-960065, info@pmcrajk.edu.pk, https://pmcrajk.edu.pk/admission.php
> - MBBS Medical College Mirpur - 05827-920453, info@mbbsmc.edu.pk
1. **Pass MDCAT** - Register at https://mdcat.pmdc.pk (PKR 9,000) and sit the test in your domicile region - SZABMU conducts it for AJK at Muzaffarabad, Rawalakot and Mirpur. You need 99/180 for MBBS or 90/180 for BDS.
2. **Compute your merit** - PM&DC formula - MDCAT 50% + HSSC 40% + SSC 10%.
3. **Apply to the AJK public colleges** - Applications for AJKMC, MBBSMC and PMC are taken online by the AJK Joint Admission Committee. Its site jac.ajk.gov.pk is not working, so follow announcements through the PMC admission cell (05824-960065), MBBSMC (05827-920453) and PM&DC. The last cycle's application fee was reported as Rs. 3,000 + Rs. 400 (secondary source).
4. **Also apply to federal colleges (SZABMU)** - AJK-domicile students compete on open merit AND on 2 AJK-quota seats each at Federal Medical College Islamabad and Shaikh Khalifa Bin Zayed College Lahore - http://www.szabmu.edu.pk/admission/mbbs-admission
5. **Merit and selection lists** - Provisional merit list, then selection lists; college preferences cannot be changed after submission.
6. **Join the college** - Verify original documents, pay the fee and register with PM&DC (every admitted student must be registered).
**Documents:**
- [ ] SSC certificate / result card (O-level needs IBCC equivalence)
- [ ] HSSC (Pre-Medical) certificate / transcript (A-level needs IBCC equivalence)
- [ ] PM&DC MDCAT result certificate
- [ ] AJK State Subject certificate (Class-I) and domicile; Permanent Resident Certificate (PRC)
- [ ] Proof of residence / property / education in the domicile region (required by SZABMU for domicile quotas)
- [ ] Candidate's CNIC / Form-B and guardian's CNIC
- [ ] Passport-size photographs with blue background (8 for PMC)
- [ ] Paid application-fee challan (separate for MBBS and BDS at SZABMU)
- [ ] Disability certificate from a government disability board (disabled quota only)

## 6. Admission dates (as of 2026-09-29)

In [10]:
pd.DataFrame(advisor.upcoming_deadlines(kb, __import__("datetime").date(2026, 9, 29), include_past_days=20))[["date", "days_left", "institution", "title", "trust"]]

,date,days_left,institution,title,trust
0,2026-09-11,-18,must,MUST undergraduate Fall 2026 - application deadline (extended),official
1,2026-09-12,-17,must,"MUST initial merit list - CS, IT, Pharm-D, DPT, MLT, BBA, Engineering Techno...",official
2,2026-09-14,-15,must,MUST initial merit list - Software Engineering,official
3,2026-09-18,-11,must,MUST 1st merit list (all programs),official
4,2026-09-20,-9,pmdc,MDCAT 2026 held (rescheduled from 16 August),official
5,2026-09-25,-4,must,MUST 2nd merit list,official
6,2026-09-30,1,must,MUST 3rd merit list,official
7,2026-10-05,6,uajk,UAJK Fall 2026 admissions - last date to apply (extended),official
8,2026-10-05,6,uajk,UAJK merit scholarship / tuition rebate 2026-27 - apply by,official


## 7. The chat assistant on sample questions

In [11]:
qs = ["I did FSc pre-medical 946/1100, matric 1010/1100, MDCAT 140/180, I want MBBS or DPT",
      "FSc pre-engineering 880/1100, ECAT 210 - engineering", "ICS 64% computer science", "MDCAT passing marks",
      "ECAT total marks", "How do I apply to UAJK?", "how to apply for MBBS in AJK", "UAJK last date",
      "mujhe MUST mein admission kaise milega", "I missed the deadline, can I still apply?", "How do I get my transcript?"]
pd.DataFrame([{"question": q, **{k: v for k, v in assistant.answer(q).items() if k in ("intent", "action", "institution")}} for q in qs])

,question,action,intent,institution
0,"I did FSc pre-medical 946/1100, matric 1010/1100, MDCAT 140/180, I want MBBS...",GUIDE,admission_advice,NaN
1,"FSc pre-engineering 880/1100, ECAT 210 - engineering",GUIDE,admission_advice,NaN
2,ICS 64% computer science,GUIDE,admission_advice,NaN
3,MDCAT passing marks,GUIDE,test_info,NaN
4,ECAT total marks,GUIDE,test_info,NaN
5,How do I apply to UAJK?,GUIDE,how_to_apply,uajk
6,how to apply for MBBS in AJK,GUIDE,how_to_apply,NaN
7,UAJK last date,ALERT,deadlines,uajk
8,mujhe MUST mein admission kaise milega,GUIDE,how_to_apply,must
9,"I missed the deadline, can I still apply?",ESCALATE,deadlines,NaN


In [12]:
display(Markdown(assistant.answer("MDCAT passing marks")["answer"]))

### MDCAT (Medical & Dental Colleges Admission Test)
**Total marks:** 180
**Minimum (mbbs):** 55% = 99 / 180
**Minimum (bds):** 50% = 90 / 180
**Note:** Session 2025-26 only: PM&DC allowed up to 3% lower (MBBS 52% = 94 marks, BDS 47% = 85 marks) ONLY to fill seats left vacant after all eligible candidates (notification 8 April 2026)
**Format:** 180 MCQs, 1 mark each, NO negative marking - Biology 81, Chemistry 45, Physics 36, English 9, Logical reasoning 9; paper-based, in English
**Conducted for ajk by:** SZABMU Islamabad - test centres in Muzaffarabad, Rawalakot and Mirpur (you must test in your domicile region)
**Fee:** PKR 9,000 (late PKR 13,000); international centre PKR 45,000 (late 55,000) - non-refundable
**Dates 2026:** Registration 22 June - 8 July 2026 (extended to 13 July, late fee till 18 July); test rescheduled from 16 August to 20 September 2026
**Merit formula:** MDCAT 50% + HSSC 40% + SSC 10%
**Register:** https://mdcat.pmdc.pk
_Sources: https://pmdc.pk/Documents/Others/Public%20Notice%20Regarding%20Opening%20of%20Online%20Registration%20Portal%20for%20MDCAT-2026.pdf, https://pmdc.pk/Documents/law/Admissions%20Regulations-2025.pdf, https://pmdc.pk/Documents/Others/Notification%20regarding%20timeline%20Extension%20&%20Reduction%20of%20MDAT%20passing%20Percentage%20for%20Session%202025-26..pdf, https://pmdc.pk/Documents/press/PRESS%20RELEASE-MDCAT%20Rescheduled.pdf_

## 8. The MCP server, through the official MCP client

In [13]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import tempfile

async def mcp_demo():
    params = StdioServerParameters(command=sys.executable, args=[str(ROOT / "mcp_server" / "server.py")],
                                   env={**os.environ, "JK_DISABLE_LLM": "1", "PYTHONIOENCODING": "utf-8"})
    with open(Path(tempfile.gettempdir()) / "jk_mcp_server.log", "w") as errlog:  # Jupyter's stderr has no OS handle
        async with stdio_client(params, errlog=errlog) as (r, w):
            async with ClientSession(r, w) as s:
                await s.initialize()
                print("tools:", [t.name for t in (await s.list_tools()).tools])
                res = await s.call_tool("check_admission", {"group": "pre_med", "hssc_obtained": 946, "hssc_total": 1100,
                                                            "ssc_obtained": 1010, "ssc_total": 1100, "mdcat_obtained": 140, "interests": "medicine"})
                p = json.loads(res.content[0].text)["programs"][0]
                print(p["program"], "->", p["eligibility"], "| aggregate", p["merit_aggregate"])

await mcp_demo()

tools: ['check_admission', 'get_test', 'how_to_apply', 'admission_dates', 'list_programs', 'search_admission_sources', 'ask_jk_assistance']
MBBS - AJK Medical College Muzaffarabad / MBBS Medical College Mirpur / Poonch Medical College Rawalakot -> eligible | aggregate 82.472
